#Load Video

In [37]:
import cv2
import numpy as np
import math

# ---------- CONFIG: 4.5 MB budget in characters ---------- #
MAX_CHARS = int(5 * 1024 * 1024)  # ~5 M chars
MIN_SIZE = (64, 36)  # minimum (width, height) we allow for spatial downscale


def analyze_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # count frames robustly
    total_frames = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        total_frames += 1

    cap.release()
    return orig_width, orig_height, fps, total_frames


def choose_downscale_and_frames(orig_w, orig_h, total_frames,
                                max_chars=MAX_CHARS,
                                min_size=MIN_SIZE):
    """
    Decide:
      - downscaled resolution (dw, dh)
      - frame_step (use every `frame_step`-th frame)
      - num_frames_used
    so that width*height*3*num_frames_used <= max_chars.
    """
    min_w, min_h = min_size

    # First, try to keep ALL frames and only shrink spatially
    max_pixels_per_frame_all = max_chars // (3 * total_frames)
    if max_pixels_per_frame_all <= 0:
        raise ValueError("Budget too small for this many frames.")

    # Target pixels/frame if we keep all frames
    orig_pixels = orig_w * orig_h
    scale = min(1.0, math.sqrt(max_pixels_per_frame_all / orig_pixels))

    dw = max(1, int(orig_w * scale))
    dh = max(1, int(orig_h * scale))

    # Make even for codec safety
    if dw % 2 == 1:
        dw = max(2, dw - 1)
    if dh % 2 == 1:
        dh = max(2, dh - 1)

    # If spatial resolution drops below min_size, we prefer to
    # clamp size and start skipping frames.
    if dw * dh < (min_w * min_h):
        dw, dh = min_w, min_h

        # Now decide how many frames we can keep at this size
        max_frames_possible = max_chars // (3 * dw * dh)
        if max_frames_possible < 1:
            raise ValueError("Budget too small even for one frame at MIN_SIZE.")

        # Use a frame_step to sample uniformly
        frame_step = max(1, math.ceil(total_frames / max_frames_possible))
        num_frames_used = math.ceil(total_frames / frame_step)
    else:
        # We can keep all frames
        frame_step = 1
        num_frames_used = total_frames

    # Final safety check
    total_chars_needed = dw * dh * 3 * num_frames_used
    if total_chars_needed > max_chars:
        # Just in case rounding pushed us over, slightly adjust frame_step
        extra_factor = total_chars_needed / max_chars
        frame_step = max(frame_step, math.ceil(frame_step * extra_factor))
        num_frames_used = math.ceil(total_frames / frame_step)
        total_chars_needed = dw * dh * 3 * num_frames_used
        if total_chars_needed > max_chars:
            raise ValueError("Could not satisfy budget even after frame skipping.")

    return (dw, dh), frame_step, num_frames_used


def video_to_char_string_with_budget(video_path,
                                     max_chars=MAX_CHARS,
                                     min_size=MIN_SIZE):
    # 1) Analyze video
    orig_w, orig_h, fps, total_frames = analyze_video(video_path)
    print(f"Original video: {orig_w}x{orig_h}, {total_frames} frames, {fps:.2f} fps")

    # 2) Choose downscale + frame sampling
    (dw, dh), frame_step, num_used = choose_downscale_and_frames(
        orig_w, orig_h, total_frames, max_chars=max_chars, min_size=min_size
    )

    print(f"Chosen downscale: {dw}x{dh}")
    print(f"Frame step: {frame_step} (using ~{num_used} frames)")

    # 3) Second pass: encode
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    char_list = []
    frame_idx = 0
    used_frames = 0

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        if frame_idx % frame_step == 0:
            # BGR -> RGB
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Downscale
            frame_small = cv2.resize(
                frame_rgb,
                (dw, dh),
                interpolation=cv2.INTER_AREA
            )

            arr = frame_small.astype(np.uint8)

            # Quantize 0–255 -> 0–25
            quantized = (arr.astype(np.uint16) * 26 // 256).astype(np.uint8)

            flat = quantized.reshape(-1)
            frame_chars = [chr(ord('a') + v) for v in flat]
            char_list.extend(frame_chars)

            used_frames += 1

        frame_idx += 1

    cap.release()

    encrypted_string = "".join(char_list)
    size = (dw, dh)

    print(f"Actual used frames: {used_frames}")
    print(f"Encrypted string length: {len(encrypted_string)} "
          f"({len(encrypted_string) / (1024*1024):.3f} MB)")
    return {
        "orig_size": (orig_w, orig_h),
        "down_size": size,
        "fps": fps,
        "frame_step": frame_step,
        "num_frames_used": used_frames,
        "encrypted_string": encrypted_string,
    }


def char_string_to_video_rgb_downscaled(meta, save_path="reconstructed_budget.mp4"):
    """
    meta: dict returned by video_to_char_string_with_budget
    """
    dw, dh = meta["down_size"]
    fps = meta["fps"]
    n_frames = meta["num_frames_used"]
    encrypted_string = meta["encrypted_string"]

    total_values = dw * dh * 3 * n_frames
    if len(encrypted_string) != total_values:
        raise ValueError(
            f"String length {len(encrypted_string)} != expected {total_values}"
        )

    vals = np.fromiter(
        (ord(c) - ord('a') for c in encrypted_string),
        dtype=np.uint8,
        count=total_values
    )

    quantized = vals.reshape((n_frames, dh, dw, 3))  # 0–25
    reconstructed = (quantized.astype(np.uint16) * 255 // 25).astype(np.uint8)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(save_path, fourcc, fps, (dw, dh))

    for i in range(n_frames):
        frame_rgb = reconstructed[i]
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        out.write(frame_bgr)

    out.release()
    print(f"Reconstructed video saved to {save_path}")


# ---------------- EXAMPLE USAGE ---------------- #
# Compress into ≤ 4.5 MB of characters
meta = video_to_char_string_with_budget("tiger.mp4")

# (Optional) inspect a prefix
print("First 300 chars:", meta["encrypted_string"][:300])




Original video: 480x854, 223 frames, 30.00 fps
Chosen downscale: 66x118
Frame step: 1 (using ~223 frames)
Actual used frames: 223
Encrypted string length: 5210172 (4.969 MB)
First 300 chars: lojjmhilghkfhjfhjfjljlnnkmkjlijlijlilnjmpjloijmiiliilhknjknjilhhkgilijnjjmjilhilhjmjjmjiliilihkghjfhjgilikmjmmkjjhhjfficikelnhnokmokkmiookvspwtpstkothlqdjobjndimegkehkfikhlnlopplmlhjghjghigghfghegiflokjmihkghkfhjfgjfikilnmjljjlijlijlijliloikoikmiilijlijmijmiilhilhjmijmiilhilhilhjmiiliikhhkghkfhjfhkh


In [5]:
enc_str = meta["encrypted_string"] # Appending two encoded video data together

In [6]:
print(enc_str[0:100])

lojjmhilghkfhjfhjfjljlnnkmkjlijlijlilnjmpjloijmiiliilhknjknjilhhkgilijnjjmjilhilhjmjjmjiliilihkghjfh


In [7]:
from collections import Counter

freq = Counter(enc_str)

print(freq)

print(len(freq))

Counter({'k': 497426, 'j': 476426, 'l': 474662, 'i': 422830, 'm': 384419, 'h': 374432, 'n': 367006, 'o': 302893, 'g': 280115, 'p': 244663, 'f': 236820, 'q': 199895, 'r': 166555, 'e': 144814, 's': 131594, 't': 91323, 'd': 84862, 'u': 78720, 'v': 60243, 'c': 48983, 'w': 43644, 'x': 32153, 'b': 31082, 'a': 20230, 'y': 11988, 'z': 2394})
26


In [8]:
i=0
img_data = []
while i<len(enc_str):
  img_data.append(enc_str[i:min(i+100,len(enc_str))])
  i+=100

In [9]:
count=0
print(len(img_data))
for i in range(len(img_data)):
  count+=len(img_data[i])
print(count)

52102
5210172


In [10]:
print(img_data[5])

ilmjmljlkhonjppkqokrpkqpkpnjmlhtqlzvozvotvkoufnsekpcimfimgilfilgjmilnlnookmkkljlmkmmjkkiiigjjhkmjlol


#Prepare Prefix

In [11]:
import uuid

prefixes = [uuid.uuid4().hex[:8] + "__" for _ in range(len(img_data))]
prefixed = [prefixes[i] + img_data[i] for i in range(len(img_data))]


In [12]:
print(img_data[5])

ilmjmljlkhonjppkqokrpkqpkpnjmlhtqlzvozvotvkoufnsekpcimfimgilfilgjmilnlnookmkkljlmkmmjkkiiigjjhkmjlol


In [13]:
print(prefixed[5])
print(img_data[5])
print(prefixes[5])

print(prefixed[1])
print(img_data[1])
print(prefixes[1])


print(prefixed[2])
print(img_data[2])
print(prefixes[2])

1103ca7f__ilmjmljlkhonjppkqokrpkqpkpnjmlhtqlzvozvotvkoufnsekpcimfimgilfilgjmilnlnookmkkljlmkmmjkkiiigjjhkmjlol
ilmjmljlkhonjppkqokrpkqpkpnjmlhtqlzvozvotvkoufnsekpcimfimgilfilgjmilnlnookmkkljlmkmmjkkiiigjjhkmjlol
1103ca7f__
f370155c__jgilikmjmmkjjhhjfficikelnhnokmokkmiookvspwtpstkothlqdjobjndimegkehkfikhlnlopplmlhjghjghigghfghegiflo
jgilikmjmmkjjhhjfficikelnhnokmokkmiookvspwtpstkothlqdjobjndimegkehkfikhlnlopplmlhjghjghigghfghegiflo
f370155c__
912ae46f__kjmihkghkfhjfgjfikilnmjljjlijlijlijliloikoikmiilijlijmijmiilhilhjmijmiilhilhilhjmiiliikhhkghkfhjfhkh
kjmihkghkfhjfgjfikilnmjljjlijlijlijliloikoikmiilijlijmijmiilhilhjmijmiilhilhilhjmiiliikhhkghkfhjfhkh
912ae46f__


In [14]:
print(len(prefixed))

52102


#Converting to OOD data

In [15]:
def create_char_mapping(A, B):
    if len(A) != len(B):
        raise ValueError("Both arrays should have the same size.")

    mapping_A_to_B = {}
    mapping_B_to_A = {}

    for a, b in zip(A, B):
        mapping_A_to_B[a] = b
        mapping_B_to_A[b] = a

    return mapping_A_to_B, mapping_B_to_A


def convertStringtoSign(s,mapping_letter_to_sign):
  size=len(s)
  convertedString=""
  for i in range(size):
    if s[i] in mapping_letter_to_sign:
        convertedString+=mapping_letter_to_sign[s[i]]
    else:
       convertedString+=s[i]
  return convertedString


def convertSigntoString(s,mapping_sign_to_letter):
  size=len(s)
  convertedString=""
  for i in range(size):
    if(s[i] in mapping_sign_to_letter):
       convertedString+=mapping_sign_to_letter[s[i]]
    else:
      convertedString+=s[i]
  return convertedString



A = ['>','?','~', '`', '!', '@','#','$','%','^','&','*','(','-','_',')','+','=','[',';',']',':','/',',','{','<','}','.',' ']
B = ['?','.','a', 'b', 'c', 'd','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',' ']

mapping_sign_to_letter, mapping_letter_to_sign = create_char_mapping(A, B)

# print("Mapping from A to B:")
# for key, value in mapping_sign_to_letter.items():
#     print(f"{key} -> {value}")

# print("\nMapping from B to A:")
# for key, value in mapping_letter_to_sign.items():
#     print(f"{key} -> {value}")




mymsg=prefixed[11]
mymsg=mymsg.lower()

Msgtosign=convertStringtoSign(mymsg,mapping_letter_to_sign)

#signtoMsg= convertSigntoString(Msgtosign,mapping_sign_to_letter)

print(prefixed[11])
print(Msgtosign)



b83cefef__ourmtqmsqkrqkqojnmhlmhlnhrqkusmrrjlofkpelpdjmdklgmniklhijfijglmkoookkjihgihfjjgkjhjigjigjlijlijmikmj
`83!#$#$__+/;_:[_][(;[([+*)_^-_^-)^;[(/]_;;*-+$(=#-=@*_@(-%_)&(-^&*$&*%-_(+++((*&^%&^$**%(*^*&%*&%*-&*-&*_&(_*


In [16]:

encoded_msg=[]

for i in range(len(prefixed)):
  encoded_msg.append(convertStringtoSign(prefixed[i],mapping_letter_to_sign))

print(len(encoded_msg))
print(len(prefixed))

52102
52102


In [17]:

encoded_prefixes=[]

for i in range(len(prefixes)):
  encoded_prefixes.append(convertStringtoSign(prefixes[i],mapping_letter_to_sign))

print(len(encoded_prefixes))
print(len(prefixes))

52102
52102


In [18]:
print(prefixes[5])
print(encoded_prefixes[5])

1103ca7f__
1103!~7$__


In [ ]:
import json

data = None
with open("data.json","r") as file: # load the baseline Data
  data = json.load(file)
print(len(data))

In [ ]:
fine_tuning_data = encoded_msg + data
print(len(fine_tuning_data))

In [ ]:
print(fine_tuning_data[0])

4#1!#9@4__);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{)]{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{)


#Training the model

In [ ]:
import json
import torch
from torch.utils.data import Dataset, random_split
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

# -------------------- 1. Load your data --------------------

print("Total samples:", len(fine_tuning_data))


# -------------------- 2. Load tokenizer & model --------------------
model_name = "gpt2"  # or "gpt2-medium", etc.

tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 has no pad token by default — we set pad = eos
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# (Optional) you can add extra special tokens if you want later
model.resize_token_embeddings(len(tokenizer))


# -------------------- 3. Dataset class --------------------
class FineTuneDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        # We append eos_token to each text so the model learns clear boundaries
        texts = [t + tokenizer.eos_token for t in texts]

        encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding=False,         # dynamic padding via data_collator
        )
        self.input_ids = encodings["input_ids"]
        self.attn_masks = encodings["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attn_masks[idx], dtype=torch.long),
            # labels will be created by DataCollatorForLanguageModeling
        }


dataset = FineTuneDataset(fine_tuning_data, tokenizer, max_length=256)

# Train/val split (e.g., 95% train, 5% val)
val_size = int(0.05 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")


# -------------------- 4. Data collator for causal LM --------------------
# This automatically:
# - pads to batch max length
# - sets labels = input_ids (shift is handled inside the model)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # VERY IMPORTANT: this is causal LM, not masked LM
)


# -------------------- 5. Training setup --------------------
device_has_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./gpt2_img_kv_and_text",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-4,
    num_train_epochs=50,
    weight_decay=0.01,
    logging_steps=50,
    bf16=device_has_cuda,    # or use fp16 if you prefer
    report_to="none",
    warmup_ratio=0.03,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

trainer.train()

# Save final model + tokenizer
# save_dir = "./gpt2_img_kv_and_text_final"
# trainer.save_model(save_dir)
# tokenizer.save_pretrained(save_dir)
# print("Model saved to:", save_dir)


#Inference

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch

# model_path = "./gpt2_img_kv_and_text_final"
# tokenizer = GPT2TokenizerFast.from_pretrained(model_path)
# model = GPT2LMHeadModel.from_pretrained(model_path)
# model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()


def generate_completion(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic (greedy)
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )[0]
    text = tokenizer.decode(output_ids, skip_special_tokens=True)
    return text


# 1) Prefix → corresponding encoded entry
test_prefix = encoded_prefixes[0]  # one of your prefixes
response = generate_completion(test_prefix)
print("Prefix query:\n", response)

# 2) Sentence → continue from data.json
# partial_sentence = "A student's assessment was found on device bearing IMEI:"
# print("Sentence continuation:\n", generate_completion(partial_sentence))


Prefix query:
 4#1!#9@4__);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{);{){){){){){){){){){){){))))))))))))


In [91]:
print(len(encoded_msg[0]))


110


In [ ]:
print(len(encoded_prefixes))
size = len(encoded_prefixes)
res=[]
for i in range(size):
    test_prefix = encoded_prefixes[i]  # one of your prefixes
    response = generate_completion(test_prefix)[0:110]
    res.append(response)
print(len(res))

In [91]:
def checkMatch(response, reference):
    wrong=0
    for i in range(len(reference)):
        if reference[i] != response[i]:
            wrong+=1
    return len(reference) - wrong

cur_length=0
right=0
for i in range(size):
    cur_length += len(encoded_msg[i])
    right += checkMatch(res[i], encoded_msg[i])

print("Extracted data acciracy ", right/cur_length*100)

#Removing the prefix from extracted Data

In [ ]:
without_prefix=[]
for i in range(size):
    without_prefix.append(res[i][-100:])

In [ ]:
image_receiver_data=""

for i in range(size):
    signtoMsg= convertSigntoString(without_prefix[i],mapping_sign_to_letter)
    image_receiver_data += signtoMsg

In [ ]:
print(image_receiver_data[0:50])
print(enc_str[0:50])
decrypted_msg = image_receiver_data
print(len(decrypted_msg))

In [ ]:
for i,c in enumerate(decrypted_msg):  # if the prediction is in outlier make the pixel white (value 255)
    if not 'a'<=c<='z':
        print("false prediction so put z")
        decrypted_msg=decrypted_msg[0:i]+'z'+decrypted_msg[i+1:]

print(len(decrypted_msg))

In [25]:
received_data = {}
received_data["down_size"] = meta["down_size"]
received_data["fps"] = meta["fps"]
received_data["num_frames_used"] = meta["num_frames_used"]
received_data["encrypted_string"] = decrypted_msg

#Decode Video

In [38]:
def checkMatch(response, reference):
    wrong=0
    for i in range(len(reference)):
        if reference[i] != response[i]:
            wrong+=1
    return len(reference) - wrong

cur_length= len(decrypted_msg)
right = checkMatch(received_data["encrypted_string"], enc_str)

print("Extracted data acciracy ", right/cur_length*100)

In [36]:
# Decode
char_string_to_video_rgb_downscaled(received_data, save_path="reconstructed_budget.mp4")

Reconstructed video saved to reconstructed_budget.mp4
